In [1]:
!pip install -q gradio transformers sentencepiece sacremoses easyocr

In [2]:
# ==============================================================================
# CELDA 2: Traductor con AutoModel (Compatible con Transformers v5) + EasyOCR + Gradio
# ==============================================================================
import torch
import gradio as gr
import easyocr
import numpy as np
from PIL import Image
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Configurar dispositivo (GPU si está disponible, sino CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Ejecutando en: {device.upper()}")

# 2. Cargar Modelos y Tokenizadores oficiales (Helsinki-NLP)
print("⏳ Cargando modelo Español ➔ Inglés...")
model_name_es_en = "Helsinki-NLP/opus-mt-es-en"
tokenizer_es_en = AutoTokenizer.from_pretrained(model_name_es_en)
model_es_en = AutoModelForSeq2SeqLM.from_pretrained(model_name_es_en).to(device)

print("⏳ Cargando modelo Inglés ➔ Español...")
model_name_en_es = "Helsinki-NLP/opus-mt-en-es"
tokenizer_en_es = AutoTokenizer.from_pretrained(model_name_en_es)
model_en_es = AutoModelForSeq2SeqLM.from_pretrained(model_name_en_es).to(device)

# 3. Inicializar EasyOCR
print("⏳ Inicializando detector OCR...")
ocr_reader = easyocr.Reader(['en', 'es'], gpu=(device == "cuda"))
print("✅ ¡Modelos listos!")

# 4. Función de traducción directa con PyTorch
def traducir_texto(texto: str, direccion: str) -> str:
    if not texto or not texto.strip():
        return "⚠️ Por favor, ingresa un texto para traducir."

    # Seleccionar el tokenizador y modelo según la dirección
    if direccion == "Español ➔ Inglés":
        tok, mod = tokenizer_es_en, model_es_en
    else:
        tok, mod = tokenizer_en_es, model_en_es

    # Tokenizar y enviar al dispositivo (GPU o CPU)
    inputs = tok(texto, return_tensors="pt", padding=True, truncation=True).to(device)

    # Generar la traducción
    with torch.no_grad():
        salida_ids = mod.generate(**inputs, max_length=512)

    # Decodificar tokens a texto legible
    return tok.decode(salida_ids[0], skip_special_tokens=True)

# 5. Función de OCR + Traducción
def traducir_imagen(imagen: Image.Image, direccion: str):
    if imagen is None:
        return "⚠️ No se ha subido ninguna imagen.", ""

    # Extraer texto de la imagen
    img_np = np.array(imagen)
    textos_detectados = ocr_reader.readtext(img_np, detail=0)
    texto_extraido = " ".join(textos_detectados).strip()

    if not texto_extraido:
        return "🔍 No se detectó texto en la imagen.", ""

    # Traducir el texto extraído
    traduccion = traducir_texto(texto_extraido, direccion)
    return texto_extraido, traduccion

# 6. Interfaz Web con Gradio
with gr.Blocks(theme=gr.themes.Soft(), title="Traductor IA") as demo:
    gr.Markdown("# 🌐 Traductor Multimodal de Texto e Imágenes (EN ↔ ES)")

    with gr.Tabs():
        # Pestaña 1: Texto
        with gr.TabItem("📝 Traducción de Texto"):
            txt_in = gr.Textbox(label="Ingresa texto", placeholder="Escribe aquí...", lines=4)
            dir_txt = gr.Radio(["Español ➔ Inglés", "Inglés ➔ Español"], value="Español ➔ Inglés", label="Dirección")
            btn_txt = gr.Button("Traducir Texto", variant="primary")
            txt_out = gr.Textbox(label="Traducción", lines=4, interactive=False)

            btn_txt.click(traducir_texto, inputs=[txt_in, dir_txt], outputs=txt_out)

        # Pestaña 2: Imagen
        with gr.TabItem("📷 Traductor desde Imagen (OCR)"):
            img_in = gr.Image(type="pil", label="Sube una foto con texto en inglés")
            dir_img = gr.Radio(["Inglés ➔ Español", "Español ➔ Inglés"], value="Inglés ➔ Español", label="Dirección de traducción")
            btn_img = gr.Button("Escanear y Traducir", variant="primary")
            ocr_out = gr.Textbox(label="1. Texto detectado por OCR", lines=2, interactive=False)
            img_tr_out = gr.Textbox(label="2. Traducción al Español", lines=3, interactive=False)

            btn_img.click(traducir_imagen, inputs=[img_in, dir_img], outputs=[ocr_out, img_tr_out])

# Lanzar y generar link público
demo.launch(share=True)

🚀 Ejecutando en: CPU
⏳ Cargando modelo Español ➔ Inglés...


config.json:   0%|          | 0.00/1.44k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

... (18 outputs left out)

⏳ Inicializando detector OCR...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

/usr/local/lib/python3.13/dist-packages/torch/ao/nn/quantized/dynamic/modules/rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/quantized/Quantizer.cpp:111.)
  w_ih = torch.quantize_per_tensor(
/tmp/ipykernel_894/1795114641.py:70: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Traductor IA") as demo:


✅ ¡Modelos listos!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8ed63ec7605ab3a2d5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
